# Phase 3 Experiments - Phase A: Baselines + Prune Finetuned

This notebook runs:
- Run 1.1-1.4: All baseline evaluations (T1, T2, FS1, FS2)
- Run 3.1-3.2: Pruning on finetuned students (FS1, FS2)

**IMPORTANT**: Each model uses its original eval_fold from Phase 1 & 2!
- T1: eval_fold=3
- T2: eval_fold=2
- FS1: eval_fold=2
- FS2: eval_fold=5

**Estimated Time**: 3-4 hours

**Dependencies**: None (can run immediately)

**Total Runs**: 6

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q iterative-stratification scikit-learn pandas numpy tqdm

In [ ]:
# Setup logging
import sys
from datetime import datetime

class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")
    
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger("/kaggle/working/experiment_log.txt")
print(f"Experiment started at: {datetime.now()}")
print("Phase A: Baselines + Prune Finetuned Students")

In [ ]:
# Clone repository
!git clone https://github.com/SaifSiddique009/kd_pruning_quantization_framework_for_nlp.git
%cd kd_pruning_quantization_framework_for_nlp
!git checkout phase3-comprehensive-experiments
!git log -1 --oneline

In [ ]:
# Verify dataset
import os
DATASET_PATH = "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv"

if os.path.exists(DATASET_PATH):
    import pandas as pd
    df = pd.read_csv(DATASET_PATH)
    print(f"Dataset loaded: {len(df)} samples")
    print(f"Columns: {df.columns.tolist()}")
else:
    print("ERROR: Dataset not found! Please add the dataset to your Kaggle notebook.")

---
## Scenario 1: Baselines (4 runs)
---

### Run 1.1: Teacher T1 - XLM-RoBERTa Baseline (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario1/T1_baseline

### Run 1.2: Teacher T2 - BanglaBERT Baseline (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-sagor-bangla-bert-base" \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario1/T2_baseline

### Run 1.3: Finetuned Student FS1 - SahajBERT Baseline (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario1/FS1_baseline

### Run 1.4: Finetuned Student FS2 - BanglaBERT-small Baseline (eval_fold=5)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-csebuetnlp-banglabert_small" \
    --use_original_folds \
    --eval_fold 5 \
    --output_dir ./results/scenario1/FS2_baseline

---
## Scenario 3: Prune Finetuned Students (2 runs)
---

### Run 3.1: Prune FS1 (SahajBERT-ft) with Magnitude (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --prune_method magnitude \
    --prune_sparsity 0.65 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario3/FS1_magnitude

### Run 3.2: Prune FS2 (BanglaBERT-small-ft) with Magnitude (eval_fold=5)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-csebuetnlp-banglabert_small" \
    --prune_method magnitude \
    --prune_sparsity 0.65 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 5 \
    --output_dir ./results/scenario3/FS2_magnitude

---
## Final Status Check & Save Results
---

In [ ]:
import os
import json
from datetime import datetime

print(f"\n{'='*70}")
print(f"EXPERIMENT COMPLETION STATUS - {datetime.now()}")
print(f"{'='*70}\n")

experiments = [
    # Scenario 1: Baselines
    ("1.1 T1 Baseline (XLM-RoBERTa)", "./results/scenario1/T1_baseline", 3),
    ("1.2 T2 Baseline (BanglaBERT)", "./results/scenario1/T2_baseline", 2),
    ("1.3 FS1 Baseline (SahajBERT-ft)", "./results/scenario1/FS1_baseline", 2),
    ("1.4 FS2 Baseline (BanglaBERT-small-ft)", "./results/scenario1/FS2_baseline", 5),
    # Scenario 3: Prune Finetuned
    ("3.1 FS1 + Magnitude Pruning", "./results/scenario3/FS1_magnitude", 2),
    ("3.2 FS2 + Magnitude Pruning", "./results/scenario3/FS2_magnitude", 5),
]

success_count = 0
results_summary = []

print(f"{'Experiment':<45} {'Fold':<6} {'F1 Weighted':<12} {'F1 Macro':<12} {'Status'}")
print("-" * 90)

for name, output_dir, fold in experiments:
    json_path = os.path.join(output_dir, "results_final.json")
    
    if os.path.exists(json_path):
        with open(json_path) as f:
            data = json.load(f)
        
        # Get the final stage metrics
        if isinstance(data, list):
            final_metrics = data[-1]
        elif 'metrics' in data:
            final_metrics = data['metrics'][-1]  # Get last stage (e.g., after_pruning)
        else:
            final_metrics = data
        
        f1_weighted = final_metrics.get('f1_weighted', 'N/A')
        f1_macro = final_metrics.get('f1_macro', 'N/A')
        
        if isinstance(f1_weighted, (int, float)) and isinstance(f1_macro, (int, float)):
            print(f"{name:<45} {fold:<6} {f1_weighted:<12.4f} {f1_macro:<12.4f} SUCCESS")
        else:
            print(f"{name:<45} {fold:<6} {str(f1_weighted):<12} {str(f1_macro):<12} SUCCESS")
        
        results_summary.append({
            "experiment": name, 
            "fold": fold,
            "f1_weighted": f1_weighted, 
            "f1_macro": f1_macro,
            "status": "SUCCESS"
        })
        success_count += 1
    else:
        print(f"{name:<45} {fold:<6} {'N/A':<12} {'N/A':<12} FAILED")
        results_summary.append({"experiment": name, "fold": fold, "status": "FAILED"})

print("-" * 90)
print(f"\nCompleted: {success_count}/{len(experiments)} experiments")
print(f"{'='*70}")

In [ ]:
# Copy results to Kaggle output
!cp -r ./results /kaggle/working/
!ls -la /kaggle/working/results/

In [ ]:
# Run aggregation script
!python aggregate_results.py --results_dir ./results --output /kaggle/working/phase_a_summary.csv --format all

In [ ]:
print(f"\nPhase A completed at: {datetime.now()}")
print("\nNOTE: Phase A is independent. You can run Phase B (KD) in parallel.")
print("      Phase C+D DEPENDS on Phase B completing and uploading KD models!")